# 05 예상 금리 채우기 확인

금감원 신용 구간 금리의 빈 칸 수를 세고, 빈 칸 채우기(`ml/app/clean.py` `fill_credit_rates`)의 오차를 확인한다.

- 근거: 데이터 처리 기획서 3-6, 3-11 (공시 범위 안 빈 칸만 이웃 구간 보간, 연속 2칸까지, 신협 제외, ESTIMATED), 6-2
- 2026-11-13: 채우는 법을 '신용 낮은 쪽 이웃 값 복사'에서 기획서대로 '선형 보간'으로 바꿈 (이 노트북 3번 결과)
- 오차 확인 방법: 공시된 칸을 일부러 가리고 채운 값과 실제 공시 값을 비교
- 불러오기마다 다시 실행해 오차 재확인 (기획서 3-11) / GitHub에 올리기 전에 **출력 지우기**
- 맨 아래 **4. 확인 결과 보고서**

In [ ]:
import sys
from collections import Counter
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# ml 폴더 찾기 (노트북 위치: ml/data/ipynb)
ML_DIR = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "app" / "clean.py").exists())
sys.path.insert(0, str(ML_DIR))

from app.clean import CREDIT_KEYS, CU_COMPANY, FILL_MAX, clean_fss, drop_closed, fill_credit_rates, load_tables

products = drop_closed(clean_fss(load_tables()), date.today())

# 대상: 금감원 개인신용, 사업자 (신협 묶음 제외). 공시 구간 금리 8칸 (CREDIT_KEYS 순서, 신용 높은 쪽 → 낮은 쪽)
vectors = {}
for p in products:
    if p["loan_type"] not in ("CREDIT", "BIZ") or p["company_name"] == CU_COMPANY:
        continue
    v = [None] * len(CREDIT_KEYS)
    for r in p["rates"]:
        if r["rate_kind"] == "CREDIT" and r["rate_source"] == "DISCLOSED":
            v[CREDIT_KEYS.index(r["rate_key"])] = r["rate_min"]
    vectors[(p["loan_type"], p["source_code"])] = v
print("대상 상품:", dict(Counter(k[0] for k in vectors)), "/ 연속 채우는 최대 칸 수 FILL_MAX =", FILL_MAX)

## 1. 빈 칸 세기 (기획서 3-11 건수 리포트)
- 공시 칸 수 / 공시 범위 안 구멍(연속 길이별) / 범위 밖 칸

In [ ]:
rows = []
for (loan_type, _), v in vectors.items():
    known = [i for i, x in enumerate(v) if x is not None]
    holes, run = [], 0
    if known:
        for i in range(known[0], known[-1] + 1):
            if v[i] is None:
                run += 1
            elif run:
                holes.append(run)
                run = 0
    rows.append({"상품 종류": loan_type, "공시 칸": len(known),
                 "구멍 1칸": holes.count(1), "구멍 2칸": holes.count(2), "구멍 3칸 이상": sum(h >= 3 for h in holes),
                 "범위 밖 칸": (len(v) - (known[-1] - known[0] + 1)) if known else len(v)})
counts = pd.DataFrame(rows)
print("공시 칸이 하나도 없는 상품:", dict(Counter(counts.loc[counts["공시 칸"] == 0, "상품 종류"])))
counts.groupby("상품 종류")[["공시 칸", "구멍 1칸", "구멍 2칸", "구멍 3칸 이상", "범위 밖 칸"]].sum()

## 2. 지금 채운 칸 (정제 결과의 ESTIMATED)

In [ ]:
estimated = Counter(p["loan_type"] for p in products for r in p["rates"]
                    if r["rate_kind"] == "CREDIT" and r["rate_source"] == "ESTIMATED")
print("추정으로 채운 칸:", dict(estimated), "/ 신협 묶음 추정:",
      sum(r["rate_source"] == "ESTIMATED" for p in products if p["company_name"] == CU_COMPANY for r in p["rates"]))

## 3. 오차 확인: 공시 칸을 가리고 맞혀 보기
- 바로 양옆이 공시된 칸만 가림 (공시 범위 안 구멍과 같은 모양)
- 선형 보간 = `fill_credit_rates` 그대로 / 비교: 오른쪽 이웃(2026-11-12까지 쓰던 방식), 왼쪽 이웃
- 치우침 + = 실제보다 높게 잡음 (공시 금리는 신용이 낮을수록 대체로 오름)
- 연속 칸 기준 (기획서 3-11): 칸 90%의 오차가 1.5%p 안인 길이까지 채움 → 길이 1~5로 확인

In [ ]:
def interpolate(v, hidden):
    """가린 칸을 fill_credit_rates로 채운 값 (FILL_MAX를 넘으면 같은 식으로 직접 계산)"""
    masked = [None if i in hidden else x for i, x in enumerate(v)]
    filled = {CREDIT_KEYS.index(r["rate_key"]): r["rate_min"] for r in fill_credit_rates(masked)}
    if all(h in filled for h in hidden):
        return [filled[h] for h in hidden]
    left, right = hidden[0] - 1, hidden[-1] + 1
    return [v[left] + (v[right] - v[left]) * (h - left) / (right - left) for h in hidden]


METHODS = {
    "선형 보간 (지금)": interpolate,
    "오른쪽 이웃 (이전)": lambda v, hidden: [v[hidden[-1] + 1]] * len(hidden),
    "왼쪽 이웃": lambda v, hidden: [v[hidden[0] - 1]] * len(hidden),
}

results = []
for gap in range(1, 6):
    errors = {name: [] for name in METHODS}
    for v in vectors.values():
        for start in range(1, len(v) - gap):
            hidden = list(range(start, start + gap))
            if any(v[h] is None for h in hidden) or v[start - 1] is None or v[start + gap] is None:
                continue
            for name, fill in METHODS.items():
                if gap > FILL_MAX and name != "선형 보간 (지금)":   # 3칸부터는 기준 확인용이라 보간만
                    continue
                errors[name] += [pred - v[h] for h, pred in zip(hidden, fill(v, hidden))]
    for name, e in errors.items():
        if not e:
            continue
        e = np.array(e)
        results.append({"가린 칸": gap, "방식": name, "칸 수": len(e), "평균 오차 %p": round(np.abs(e).mean(), 2),
                        "칸 90% 오차 %p": round(np.quantile(np.abs(e), 0.9), 2), "치우침 %p": round(e.mean(), 2), "높게 잡은 칸 %": round((e > 0).mean() * 100, 1),
                        "1.5%p 넘게 틀림 %": round((np.abs(e) > 1.5).mean() * 100, 1)})
pd.DataFrame(results)

In [ ]:
# 공시 금리가 신용 낮은 쪽으로 갈수록 오르는지 (이웃 공시 칸끼리)
pairs = [(v[i], v[i + 1]) for v in vectors.values() for i in range(len(v) - 1) if v[i] is not None and v[i + 1] is not None]
print("이웃 공시 칸 %d쌍 중 신용 낮은 쪽이 같거나 높음: %.1f%%" % (len(pairs), np.mean([b >= a for a, b in pairs]) * 100))

## 4. 확인 결과 보고서 (2026-11-13 실행, 금감원 2026-11-11 수집 CSV 기준)

### 4-1. 빈 칸 수 (기획서 3-11 건수 리포트)

| 상품 종류 | 상품 | 공시 칸 | 구멍 1칸 | 구멍 2칸 | 구멍 3칸 이상 | 지금 채운 칸 |
| --- | --- | --- | --- | --- | --- | --- |
| 개인신용 | 115 | 500 | 9 | 3 | 4 | 15 |
| 사업자 (신협 제외) | 1,012 | 875 | 14 | 2 | 2 | 18 |

- 구간 금리 공시가 하나도 없는 상품: 개인신용 6개(기획서는 5개, 수집일 차이), 사업자 747개 → 추정하지 않음 (기획서 3-11)
- 신협 묶음은 추정 0칸 (조합마다 달라 제외, 기획서 3-11)
- 기획서 3-11의 '개인신용 16칸, 사업자 21칸'은 문서 작성 때 데이터 기준, 지금은 15칸, 18칸

### 4-2. 채우는 법 비교 (공시 칸을 가리고 맞혀 보기)

| 가린 칸 | 방식 | 평균 오차 | 칸 90% 오차 | 치우침 | 1.5%p 넘게 틀림 |
| --- | --- | --- | --- | --- | --- |
| 1칸 (653칸) | **선형 보간 (지금)** | **0.41%p** | **1.12%p** | −0.03%p | **4.9%** |
| | 오른쪽 이웃 (11-12까지) | 0.85%p | 2.30%p | +0.66%p (72%를 높게) | 19.1% |
| | 왼쪽 이웃 | 0.83%p | 2.13%p | −0.72%p | 17.9% |
| 2칸 (804칸) | **선형 보간 (지금)** | **0.49%p** | **1.27%p** | −0.05%p | **7.5%** |
| | 오른쪽 이웃 | 1.16%p | 3.14%p | +0.98%p | 26.7% |
| | 왼쪽 이웃 | 1.14%p | 2.90%p | −1.08%p | 26.2% |
| 3칸 (642칸, 기준 확인) | 선형 보간 | 0.55%p | **1.55%p** | −0.03%p | 10.4% |
| 4칸 (452칸, 기준 확인) | 선형 보간 | 0.53%p | 1.46%p | +0.12%p | 9.7% |
| 5칸 (355칸, 기준 확인) | 선형 보간 | 0.63%p | 1.90%p | +0.13%p | 16.1% |

- 선형 보간 1칸 오차 0.41%p는 기획서 3-11의 '보간 0.40%p'와 같음 → 기획서 근거가 지금 데이터로도 다시 나옴
- 이웃 값 복사는 오차가 두 배이고 한쪽으로 치우침. 공시 금리는 신용이 낮을수록 대체로 오르므로(이웃 공시 칸 967쌍 중 83.9%) 오른쪽 이웃 복사는 거의 늘 높게 잡음
- 그래서 2026-11-13에 `fill_credit_rates`를 선형 보간으로 바꿈 (대상, 범위, 연속 2칸까지, ESTIMATED 표시는 그대로)

### 4-3. 연속 칸 기준과 남은 일

- **연속 2칸까지 채우는 기준은 맞음**: 기획서 3-11 '3개 이상은 오차 1.5%p 넘음'은 칸 90%의 오차로 보면 지금 데이터와 같음 (2칸 1.27%p → 3칸 1.55%p에서 처음 1.5%p 넘음). 평균 오차(3칸 0.55%p)로는 이 기준이 안 보임
- 3칸까지 넓히면 더 채워지는 것은 3개 상품 9칸이고, 그중 8칸은 BAND5(이미 13번 칸 값으로 예상 금리가 나옴). 새로 생기는 화면 값은 사업자 1개 상품의 BAND4 1칸뿐이라 2칸 유지 (2026-11-13 결정)
- 4칸 이상 구멍 3곳 중 2곳은 신용이 낮은 쪽 금리가 더 낮은 특이한 상품이라 채우지 않는 편이 안전
- **이미 DB에 있는 추정 33칸**: 첫 적재(2026-11-12) 때 이전 방식(오른쪽 이웃) 값으로 들어감 → 다음 적재(불러오기) 때 이 방식으로 바뀜 (2026-11-13 결정)
- 추정 값은 화면에 '추정'과 '실제 금리는 기관 심사로 정해짐' 안내가 같이 나감 (기획서 4-1 문구 1)